## 2. Load & Explore the Dataset

In [1]:
SELECT *
FROM patient_insurance_dataset.csv

,PatientID,Age,Gender,State,City,Height_cm,Weight_kg,BMI,Insurance_Type,Primary_Condition,Num_Chronic_Conditions,Annual_Visits,Avg_Billing_Amount,Last_Visit_Date,Days_Since_Last_Visit,Preventive_Care_Flag
0,P10000,64,Male,GA,Unknown,151,115,50.4,Private,Arthritis,3,7,2995.0,2025-07-18 00:00:00+00:00,186,0
1,P10001,59,Male,OH,Unknown,189,68,19.0,Medicare,Depression,1,8,1209.0,2025-12-12 00:00:00+00:00,39,0
2,P10002,58,Female,PA,Unknown,156,91,37.4,Private,Asthma,1,4,999.0,2025-09-16 00:00:00+00:00,126,0
3,P10003,43,Female,GA,Unknown,152,92,39.8,Medicare,Hypertension,1,6,5638.5,2025-04-09 00:00:00+00:00,286,1
4,P10004,53,Female,NC,Unknown,167,51,18.3,Medicaid,Asthma,1,4,5796.0,2025-03-07 00:00:00+00:00,319,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,P11995,37,Male,IL,Springfield,195,109,28.7,Medicaid,None,0,4,4581.0,2025-09-20 00:00:00+00:00,122,1
1996,P11996,32,Female,NY,Rochester,151,94,41.2,Private,None,0,1,329.0,2025-08-23 00:00:00+00:00,150,1
1997,P11997,50,Female,NC,Unknown,149,115,51.8,Private,Anxiety,1,8,4942.5,2025-05-23 00:00:00+00:00,242,1
1998,P11998,74,Female,NC,Unknown,165,97,35.6,Medicare,Heart Disease,3,7,4700.0,2025-07-09 00:00:00+00:00,195,1


Prompt: YOUR PROMPT GOES HERE

## 3. Data Cleaning

One of the things necessary is to determine whether you have missing values. Handling missing values is always difficult. If only a few observations are missing, they can be excluded. If there is a significant number of missing values, several options are available depending on the type of missing value.

But here's the catch: missing values don't always show up as `NaN`. Sometimes they hide as strings like `"Unknown"`, `"None"`, `"N/A"`, or even blank spaces. The AI or even Pandas won't flag these automatically. You need to check for them yourself.

Prompt: YOUR PROMPT GOES HERE

The AI tells us that the dataset doesn't have missing values. Is that correct?

You should re-check the actual values that you saw when we count the values in each column. Sometimes "missing" data hides behind placeholder strings. 

If you look at the **City** column there are values recorded as "Unknown". Also, the **Primary condition** has "None" as a value. Pandas didn't flag these as missing because technically they're valid strings. But "Unknown" is not a real city. It's a disguised missing value.

This is a very common problem in real-world data, especially in healthcare. Missing data is sometimes filled with placeholder text like "Unknown", "None", "Not Specified", or "N/A" instead of leaving them as actual null values.

**The AI missed this.** It only checked for `NaN` values.

Prompt: YOUR PROMPT GOES HERE

Now you can see the true picture of missing data. The City column has ~50% missing, which is very high to impute reliably.

Also, you can observe that the Primary Condition for 495 patients is missing. On a deeper analysis, you can see that when the primary condition is present it has a disease or health condition associated.

Prompt: YOUR PROMPT GOES HERE

## 4. Exploratory Data Analysis


It is important you should get descriptive statistics and distribution of each variable of interest. Looking into distributions and descriptive statistics allows us to understand our patient population and determine if we need to perform any data transformation before clustering.

- For numerical variables, you need visualizations that helps you understand the spread and potential outliers in each feature. Also, you need to understand correlations.

- For categorical variables, you need counts that gives you insight into the distribution of these categories in the dataset.

Prompt: YOUR PROMPT GOES HERE

Prompt: YOUR PROMPT GOES HERE

The correlation heatmap shows how numeric features relate to each other. This helps you understand which features move together and whether any are redundant. Highly correlated features can bias clustering by effectively double-counting the same information.

The correlation heatmap reveals important relationships:

- **Age and Num_Chronic_Conditions** are strongly correlated (r = 0.80). Older patients tend to accumulate more chronic conditions.
- **Weight_kg and BMI** are strongly correlated (r = 0.84), and **Height_cm and BMI** show moderate negative correlation (r = -0.54). Since BMI is derived from height and weight, we will keep only BMI to avoid redundancy.
- **Age, Num_Chronic_Conditions, Annual_Visits, and Avg_Billing_Amount** form a cluster of moderate positive correlations (r = 0.34-0.43). Older, sicker patients visit more and cost more.
- **Days_Since_Last_Visit** shows near-zero correlation with everything else, it captures an independent dimension of patient behavior.

## 5. Clustering

Let's ask the AI to cluster our patients. A natural first instinct, and what most AI assistants will suggest, is **K-Means**.

Prompt: YOUR PROMPT GOES HERE

Prompt: YOUR PROMPT GOES HERE

Prompt: YOUR PROMPT GOES HERE

Prompt: YOUR PROMPT GOES HERE

K-Means is the most popular clustering algorithm. K-Means is elegant and fast. But it has a fundamental limitation: it uses Euclidean distance, which only works with numeric data.

In your case, K-Means only found 3 clusters, and the elbow method picked k=2 as "best." But does that really mean your patients only fall into 3 groups?

No, here's what went wrong:

**One-hot encoding**: Converted 3 categorical columns into ~15 binary columns, inflating dimensionality |
**Euclidean distance**: Treats every binary column equally — a 0/1 difference in "Gender_Male" weighs the same as a real numeric difference in Age
**Categorical similarity lost**: "Private" vs "Medicare" insurance are treated as completely unrelated, when they actually carry meaningful similarity structure
**Sparse high-dimensional space**: K-Means struggles in high-dimensional sparse spaces — centroids become meaningless.

We need a distance measure that handles mixed data. Gower distance (1971) was designed precisely for this problem. It computes a distance between 0 and 1 for each feature, using the appropriate method for each data type: 

- **Numeric features:** normalized Manhattan distance (range-scaled)
- **Categorical features:** simple matching (0 if same, 1 if different)
- **Final distance:** weighted average across all features

This means each feature contributes equally regardless of type, and we don't need to use one-hot encoding. 

With a proper distance matrix in hand, we can use hierarchical (agglomerative) clustering using **weighted linkage**, which works directly on a precomputed distance matrix. 

Hierarchical clustering:

- Works with any distance matrix: including our Gower distances.
- Does not require specifying the number of clusters upfront.
- Produces a dendrogram: a tree that shows how patients merge into clusters step by

Prompt: YOUR PROMPT GOES HERE

Prompt: YOUR PROMPT GOES HERE

Prompt: YOUR PROMPT GOES HERE

The dendrogram shows how patients are merged into clusters at each step. The height of each merge represents the distance between the groups being combined. You can "cut" the tree at different heights to get different numbers of clusters.

For understanding what each cluster means, you need to look at the average numeric features and the most common categorical values per cluster to build patient profiles as it showed above.

Prompt: YOUR PROMPT GOES HERE

## 6. Patient Segment Narratives

Based on the cluster profiles above, here are the patient segments you discovered. Let's generate a clear summary for each cluster that a non-technical stakeholder could understand: this is what makes clustering actionable.

Prompt: YOUR PROMPT GOES HERE

## 7. Predicting the Segment for a New Patient

The clustering is done and our segments are defined. But here's a real-world question: **what happens when a new patient enrolls tomorrow?** 

You need to be able to assign them to the right segment without re-running the entire clustering.

The idea is simple: compute the Gower distance from the new patient to every existing patient, then see which cluster they're closest to on average.

Prompt: YOUR PROMPT GOES HERE

## Key Takeaways 

1. **Always explore your data first** — distributions, missing values, and correlations tell you what preprocessing is needed.

2. **AI-assisted coding accelerates your workflow** but your domain knowledge is irreplaceable. You need to validate and sometimes correct the AI.

3. **Algorithm choice matters**: K-Means with one-hot encoding failed because Euclidean distance doesn't handle mixed data well. Gower distance was the right tool for this dataset.

4. **Clustering is only useful if you can interpret it**: the real value is in the patient segment narratives that inform clinical and business decisions.

5. **Practice!**: The best way to learn is by doing.